In [ ]:
from google.colab import files
files.upload()
print("Dataset Uploaded Successfully")

In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras import layers
from tensorflow.keras.applications import EfficientNetB3
from tensorflow.keras.utils import image_dataset_from_directory
from tensorflow.keras import optimizers, losses, callbacks
import matplotlib.pyplot as plt
import numpy as np
import os

In [ ]:
import zipfile

zip_ref = zipfile.ZipFile('DFU.zip', 'r')
zip_ref.extractall('/content/')
zip_ref.close()

FileNotFoundError: [Errno 2] No such file or directory: 'DFU.zip'

In [ ]:
TRAIN_PATH = '/content/DFU/Patches'
TEST_PATH  = '/content/DFU/TestSet'

In [ ]:
BATCH_SIZE = 32
IMAGE_HEIGHT = 224
IMAGE_WIDTH = 224
NUM_CLASSES = 2
EPOCHS = 50

In [ ]:
train_ds = image_dataset_from_directory(
    TRAIN_PATH,
    labels="inferred",
    validation_split=0.2,
    batch_size=BATCH_SIZE,
    image_size=(IMAGE_WIDTH, IMAGE_HEIGHT),
    subset="training",
    seed=42
)

validation_ds = image_dataset_from_directory(
    TRAIN_PATH,
    labels="inferred",
    validation_split=0.2,
    batch_size=BATCH_SIZE,
    image_size=(IMAGE_WIDTH, IMAGE_HEIGHT),
    subset="validation",
    seed=42
)

Found 1055 files belonging to 2 classes.
Using 844 files for training.
Found 1055 files belonging to 2 classes.
Using 211 files for validation.


In [ ]:
class_names = train_ds.class_names
print(f"Class Names: {class_names}")

img_augmentation = Sequential([
    layers.RandomRotation(0.15),
    layers.RandomTranslation(0.1, 0.1),
    layers.RandomFlip("horizontal"),
    layers.GaussianNoise(0.1),
    layers.RandomContrast(0.1),
], name="img_augmentation")

Class Names: ['Abnormal(Ulcer)', 'Normal(Healthy skin)']


In [ ]:
def decode_img(img):
    img = tf.io.decode_jpeg(img, channels=3)
    img = tf.image.resize(img, [IMAGE_HEIGHT, IMAGE_WIDTH])
    return img

def process_path(file_path):
    img = tf.io.read_file(file_path)
    return decode_img(img)

In [ ]:
VALID_FORMATS = ('.jpg', '.jpeg', '.png', '.gif', '.bmp')
test_files = [f for f in os.listdir(TEST_PATH) if f.lower().endswith(VALID_FORMATS)]
test_ds = tf.data.Dataset.from_tensor_slices([os.path.join(TEST_PATH, f) for f in test_files])
test_ds = test_ds.map(process_path, num_parallel_calls=tf.data.AUTOTUNE)

In [ ]:
AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.cache().prefetch(buffer_size=AUTOTUNE)
validation_ds = validation_ds.cache().prefetch(buffer_size=AUTOTUNE)


lr_callback = callbacks.ReduceLROnPlateau(monitor="val_accuracy", factor=0.1, patience=5)
stop_callback = callbacks.EarlyStopping(monitor="val_accuracy", patience=8)

In [ ]:
def build_model():
    inputs = layers.Input(shape=(IMAGE_WIDTH, IMAGE_HEIGHT, 3))
    x = img_augmentation(inputs)  # Apply Augmentation
    base_model = EfficientNetB3(include_top=False, weights="imagenet", input_tensor=x)
    x = layers.GlobalAveragePooling2D()(base_model.output)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.4)(x)
    outputs = layers.Dense(NUM_CLASSES, activation="softmax")(x)

    model = tf.keras.Model(inputs, outputs, name="EfficientNetB3")
    model.compile(
        optimizer=optimizers.Adam(learning_rate=1e-3),
        loss=losses.SparseCategoricalCrossentropy(),
        metrics=["accuracy"]
    )

    return model

In [ ]:
model = build_model()
history = model.fit(
    train_ds, validation_data=validation_ds,
    callbacks=[lr_callback, stop_callback],
    epochs=EPOCHS, verbose=1
)

In [ ]:
import matplotlib.pyplot as plt

# Accuracy
plt.plot(history.history['accuracy'], label='Train Accuracy')
plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
plt.legend()
plt.title("Model Accuracy")
plt.show()

# Loss
plt.plot(history.history['loss'], label='Train Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.legend()
plt.title("Model Loss")
plt.show()

In [ ]:
print("Final Training Accuracy:", history.history['accuracy'][-1])
print("Final Validation Accuracy:", history.history['val_accuracy'][-1])

In [ ]:
loss, accuracy = model.evaluate(validation_ds)

print(f"Test Accuracy: {accuracy * 100:.2f}%")

In [ ]:
model.save("dfu_model.h5")

In [ ]:
from google.colab import files
files.download("dfu_model.h5")

In [ ]:
#testing on training set
plt.figure(figsize=(10, 10))
for images, labels in train_ds.take(2):
    images, labels = images.numpy(), labels.numpy()
    predictions = np.argmax(model.predict(images), axis=1)

    for i in range(9):
        ax = plt.subplot(3, 3, i + 1)
        plt.imshow(images[i].astype("uint8"))
        plt.title(f"Pred: {class_names[predictions[i]]}\nActual: {class_names[labels[i]]}")
        plt.axis("off")
plt.show()

In [ ]:
#Testing on Test Set
plt.figure(figsize=(10, 10))
i = 0
for image in test_ds.take(9):
    image = image.numpy().astype("uint8")
    prediction = np.argmax(model.predict(tf.expand_dims(image, axis=0)), axis=1)

    ax = plt.subplot(3, 3, i + 1)
    plt.imshow(image)
    plt.title(f"Pred: {class_names[prediction[0]]}")
    plt.axis("off")

    i += 1
plt.show()

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report

y_pred = np.argmax(model.predict(validation_ds), axis=1)
y_true = np.concatenate([y for x, y in validation_ds], axis=0)

print(confusion_matrix(y_true, y_pred))
print(classification_report(y_true, y_pred))